In [1]:
#step 1 data import
import os
import json
import numpy as np
import pandas as pd
import neurokit2 as nk
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# ── BIDS CONFIG ───────────────────────────────────────────────────────────────
bids_root    = Path(r"C:\Users\alisa\OneDrive\Desktop\HRV_Biotrace\bids_data")
task_name    = "BBSIG"
datatype_name = "beh"

# All participants and both sessions
participant_ids = [
    "001","002","003","004","005","006","007","008","009","010",
    "011","012","015","018","019","020","021","022","023","024","025"
]
session_ids = ["01", "02"]  # ses-01 = one condition, ses-02 = other condition

# ── OPTIONAL STEPS ────────────────────────────────────────────────────────────
compute_hrv_time = True   # time-domain: SDNN, RMSSD, pNN50 etc.
compute_hrv_freq = True   # frequency-domain: LF, HF, LF/HF etc.
compute_hrv_nl   = True   # non-linear: SD1, SD2, SampEn etc.
show_plots       = False  # set True only if inspecting one participant
crop_window      = True   # we will crop per phase (pre/early/late/post)

# ── DATA IMPORT SETTINGS ──────────────────────────────────────────────────────
bbsig_preproc = True   # using our ecg-preproc.json files
rri_unit      = 's'    # RR intervals saved in seconds by BBSIG
physio_type   = 'ecg'

# rr_type: which rpeaks to use — 'original' for most, 'corrected' for sub-005/011
# We handle this per-participant in the load function below

# ── FREQUENCY DOMAIN SETTINGS ─────────────────────────────────────────────────
if compute_hrv_freq:
    interpolation_freq = 4  # Hz, standard for HRV frequency analysis

# ── STEP 1: Define data loading function ──────────────────────────────────────
def load_rr_data(bids_root, subj_id, ses_id, physio_type='ecg'):
    """
    Load RR intervals from ecg-preproc.json.
    Automatically uses corrected rpeaks if available, otherwise original.
    Returns dict with RRI (ms) and RRI_Time (ms cumulative timestamps).
    """
    base_name = f"sub-{subj_id}_ses-{ses_id}_task-{task_name}_physio"
    json_path = bids_root / "derivatives" / f"{physio_type}-preproc" / \
                f"sub-{subj_id}" / f"ses-{ses_id}" / f"{base_name}_ecg-preproc.json"

    if not json_path.exists():
        print(f"  ⚠ No ecg-preproc.json found for sub-{subj_id}/ses-{ses_id}, skipping.")
        return None

    with open(json_path) as f:
        data = json.load(f)

    # Pick best available RR series: corrected > original
    rr_s = data.get('rr_s', {})
    if 'RR_s_corrected' in rr_s:
        rr_array = np.array(rr_s['RR_s_corrected'])
        rr_label = 'corrected'
    elif 'RR_s_original' in rr_s:
        rr_array = np.array(rr_s['RR_s_original'])
        rr_label = 'original'
    else:
        print(f"  ⚠ No RR series found in JSON for sub-{subj_id}/ses-{ses_id}, skipping.")
        return None

    # Convert seconds → milliseconds
    rr_ms   = rr_array * 1000
    rr_time = np.cumsum(rr_ms)  # timestamps in ms

    print(f"  Loaded sub-{subj_id}/ses-{ses_id} [{rr_label}]: {len(rr_ms)} RR intervals")

    return {
        'RRI':      rr_ms,
        'RRI_Time': rr_time,
        'rr_label': rr_label,
        'subj_id':  f"sub-{subj_id}",
        'ses_id':   f"ses-{ses_id}"
    }



In [6]:
#quick test on one participant
test = load_rr_data(bids_root, "001", "01")
if test:
    print(f"\n  RRI range: {test['RRI'].min():.1f} – {test['RRI'].max():.1f} ms")
    print(f"  Total duration: {test['RRI_Time'][-1]/1000/60:.2f} min")

  Loaded sub-001/ses-01 [original]: 2278 RR intervals

  RRI range: 464.8 – 953.1 ms
  Total duration: 29.88 min


In [10]:
#step 2 ceopping by phase
import json
import numpy as np
from pathlib import Path

# Instead of one fixed window, we crop per phase (pre/early/late/post)
# using the phase boundaries saved in your phases JSON files.
# Returns a dict of cropped RRI per phase for one participant.

def load_phase_boundaries(bids_root, subj_id, ses_id, task_name="BBSIG"):
    """Load phase start/end times (in seconds) from phases JSON."""
    base_name   = f"sub-{subj_id}_ses-{ses_id}_task-{task_name}_physio"
    phases_path = bids_root / "derivatives" / "phases" / \
                  f"sub-{subj_id}" / f"ses-{ses_id}" / f"{base_name}_phases.json"

    if not phases_path.exists():
        print(f"  ⚠ No phases file found for sub-{subj_id}/ses-{ses_id}")
        return None

    with open(phases_path) as f:
        phase_data = json.load(f)

    return phase_data['phases'], phase_data.get('condition', 'unknown')


def crop_rr_by_phase(rrs_dict, phase_start_s, phase_end_s):
    """
    Crop RRI dict to a phase window defined by start/end in seconds.
    RRI_Time is in milliseconds, so we convert boundaries accordingly.
    Returns cropped dict or None if too few beats.
    """
    rri      = rrs_dict['RRI']        # ms
    rri_time = rrs_dict['RRI_Time']   # ms cumulative

    start_ms = phase_start_s * 1000
    end_ms   = phase_end_s   * 1000

    mask = (rri_time >= start_ms) & (rri_time <= end_ms)
    rri_cropped  = rri[mask]
    time_cropped = rri_time[mask]

    if len(rri_cropped) < 10:  # minimum beats needed for HRV
        return None

    return {
        'RRI':          rri_cropped,
        'RRI_Time':     time_cropped,
        'window_start': phase_start_s,
        'window_end':   phase_end_s,
        'n_beats':      len(rri_cropped),
        'duration_min': (time_cropped[-1] - time_cropped[0]) / 1000 / 60
    }


In [9]:
#Quick test on sub-001/ses-01
test_rr     = load_rr_data(bids_root, "001", "01")
test_phases, test_condition = load_phase_boundaries(bids_root, "001", "01")

print(f"\nCondition: {test_condition}")
print(f"{'─'*50}")

for phase_name in ['pre', 'early', 'late', 'post']:
    phase       = test_phases[phase_name]
    start_s     = phase['start']
    end_s       = phase['end']
    cropped     = crop_rr_by_phase(test_rr, start_s, end_s)

    if cropped:
        mean_hr = 60000 / cropped['RRI'].mean()
        print(f"  {phase_name.upper():6s}: {start_s:.0f}–{end_s:.0f}s | "
              f"{cropped['n_beats']} beats | "
              f"{cropped['duration_min']:.2f} min | "
              f"Mean HR: {mean_hr:.1f} bpm")
    else:
        print(f"  {phase_name.upper():6s}: ⚠ Too few beats to analyse")

  Loaded sub-001/ses-01 [original]: 2278 RR intervals

Condition: control
──────────────────────────────────────────────────
  PRE   : 157–337s | 240 beats | 2.99 min | Mean HR: 80.1 bpm
  EARLY : 367–517s | 187 beats | 2.49 min | Mean HR: 74.8 bpm
  LATE  : 1303–1483s | 221 beats | 3.00 min | Mean HR: 73.4 bpm
  POST  : 1513–1693s | 245 beats | 2.99 min | Mean HR: 81.6 bpm


In [11]:
# step 3,4,5,6,7: frequency and time domains calculated for the continuous hrv  
import json
import numpy as np
import pandas as pd
import neurokit2 as nk
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# ── CONFIG ────────────────────────────────────────────────────────────────────
bids_root    = Path(r"C:\Users\alisa\OneDrive\Desktop\HRV_Biotrace\bids_data")
task_name    = "BBSIG"
participant_ids = [
    "001","002","003","004","005","006","007","008","009","010",
    "011","012","015","018","019","020","021","022","023","024","025"
]
session_ids  = ["01", "02"]
interpolation_freq = 4  # Hz for frequency-domain

# ── HRV COMPUTATION FUNCTIONS ─────────────────────────────────────────────────

def compute_hrv_all(rrs_dict, subj_id, ses_id, condition, window='full',
                    phase=None, interp_freq=4):
    """
    Compute time, frequency and non-linear HRV on a given RRI dict.
    Returns one flat dict with all metrics + metadata.
    """
    rri      = rrs_dict['RRI']        # ms
    rri_time = rrs_dict['RRI_Time']   # ms cumulative timestamps

    # Metadata row
    row = {
        'subj_id':    subj_id,
        'ses_id':     ses_id,
        'condition':  condition,
        'window':     window,
        'phase':      phase if phase else 'full',
        'n_beats':    len(rri),
        'duration_min': (rri_time[-1] - rri_time[0]) / 1000 / 60
    }

    try:
        # ── Time-domain ───────────────────────────────────────────────────────
        hrv_time = nk.hrv_time(
            peaks    = rri_time / 1000,  # neurokit expects seconds
            sampling_rate = 1,           # we pass timestamps directly
            show     = False
        )
        row['MeanRR']  = rri.mean()
        row['MeanHR']  = 60000 / rri.mean()
        row['SDNN']    = hrv_time['HRV_SDNN'].values[0]
        row['RMSSD']   = hrv_time['HRV_RMSSD'].values[0]
        row['pNN50']   = hrv_time['HRV_pNN50'].values[0]
    except Exception as e:
        print(f"    ⚠ Time-domain failed: {e}")
        for k in ['MeanRR','MeanHR','SDNN','RMSSD','pNN50']:
            row[k] = np.nan

    try:
        # ── Frequency-domain ──────────────────────────────────────────────────
        hrv_freq = nk.hrv_frequency(
            peaks         = rri_time / 1000,
            sampling_rate = 1,
            ulf           = (0, 0.0033),
            vlf           = (0.0033, 0.04),
            lf            = (0.04, 0.15),
            hf            = (0.15, 0.4),
            show          = False
        )
        row['VLF']   = hrv_freq['HRV_VLF'].values[0]
        row['LF']    = hrv_freq['HRV_LF'].values[0]
        row['HF']    = hrv_freq['HRV_HF'].values[0]
        row['LF_HF'] = hrv_freq['HRV_LFHF'].values[0]
        row['LFn']   = hrv_freq['HRV_LFn'].values[0]
        row['HFn']   = hrv_freq['HRV_HFn'].values[0]
        row['LnHF']  = hrv_freq['HRV_LnHF'].values[0]
    except Exception as e:
        print(f"    ⚠ Frequency-domain failed: {e}")
        for k in ['VLF','LF','HF','LF_HF','LFn','HFn','LnHF']:
            row[k] = np.nan

    try:
        # ── Non-linear ────────────────────────────────────────────────────────
        hrv_nl = nk.hrv_nonlinear(
            peaks         = rri_time / 1000,
            sampling_rate = 1,
            show          = False
        )
        row['SD1']    = hrv_nl['HRV_SD1'].values[0]
        row['SD2']    = hrv_nl['HRV_SD2'].values[0]
        row['SD1_SD2']= hrv_nl['HRV_SD1SD2'].values[0]
        row['SampEn'] = hrv_nl['HRV_SampEn'].values[0]
        row['ApEn']   = hrv_nl['HRV_ApEn'].values[0]
    except Exception as e:
        print(f"    ⚠ Non-linear failed: {e}")
        for k in ['SD1','SD2','SD1_SD2','SampEn','ApEn']:
            row[k] = np.nan

    return row

# ── MAIN LOOP: FULL RECORDING ─────────────────────────────────────────────────
print("="*60)
print("HRV ANALYSIS — FULL RECORDING")
print("="*60)

all_hrv_full = []

for subj_id in participant_ids:
    for ses_id in session_ids:

        print(f"\nsub-{subj_id}/ses-{ses_id}")

        # Load RR intervals
        rrs = load_rr_data(bids_root, subj_id, ses_id)
        if rrs is None:
            continue

        # Load condition label
        phases, condition = load_phase_boundaries(bids_root, subj_id, ses_id)
        if phases is None:
            continue

        # Compute HRV on full recording
        row = compute_hrv_all(
            rrs_dict  = rrs,
            subj_id   = f"sub-{subj_id}",
            ses_id    = f"ses-{ses_id}",
            condition = condition,
            window    = 'full'
        )
        all_hrv_full.append(row)
        print(f"  ✅ RMSSD={row['RMSSD']:.2f}ms | HF={row['HF']:.4f} | condition={condition}")

# ── SAVE OUTPUT ───────────────────────────────────────────────────────────────
out_dir = bids_root / "derivatives" / "hrv-analysis"
out_dir.mkdir(parents=True, exist_ok=True)

df_full = pd.DataFrame(all_hrv_full)
out_path = out_dir / f"task-{task_name}_hrv_full_recording.tsv"
df_full.to_csv(out_path, sep='\t', index=False)

print(f"\n{'='*60}")
print(f"✅ Saved: {out_path.name}")
print(f"   {len(df_full)} rows ({len(participant_ids)} participants × {len(session_ids)} sessions)")
print(f"{'='*60}")
print(df_full[['subj_id','ses_id','condition','MeanHR','RMSSD','HF','SD1']].to_string(index=False))

HRV ANALYSIS — FULL RECORDING

sub-001/ses-01
  Loaded sub-001/ses-01 [original]: 2278 RR intervals
  ✅ RMSSD=32.20ms | HF=0.0010 | condition=control

sub-001/ses-02
  Loaded sub-001/ses-02 [original]: 2024 RR intervals
  ✅ RMSSD=30.27ms | HF=0.0013 | condition=experimental

sub-002/ses-01
  Loaded sub-002/ses-01 [original]: 2234 RR intervals
  ✅ RMSSD=57.55ms | HF=0.0042 | condition=experimental

sub-002/ses-02
  Loaded sub-002/ses-02 [original]: 1911 RR intervals
  ✅ RMSSD=67.27ms | HF=0.0149 | condition=control

sub-003/ses-01
  Loaded sub-003/ses-01 [original]: 1988 RR intervals
  ✅ RMSSD=41.40ms | HF=0.0027 | condition=control

sub-003/ses-02
  Loaded sub-003/ses-02 [original]: 1874 RR intervals
  ✅ RMSSD=39.06ms | HF=0.0026 | condition=experimental

sub-004/ses-01
  Loaded sub-004/ses-01 [original]: 1740 RR intervals
  ✅ RMSSD=49.28ms | HF=0.0014 | condition=experimental

sub-004/ses-02
  Loaded sub-004/ses-02 [original]: 1783 RR intervals
  ✅ RMSSD=37.17ms | HF=0.0016 | conditio

In [12]:
# checking the ranges, if anything is missing
print("FULL RECORDING HRV")
print("="*60)

df_check = pd.DataFrame(all_hrv_full)

# 1. Check both conditions are present for each participant
print("\n1. Condition balance:")
print(df_check.groupby(['subj_id','condition']).size().unstack(fill_value=0))

# 2. Check HRV ranges are physiologically plausible
print("\n2. Key metric ranges:")
print(f"   MeanHR:  {df_check['MeanHR'].min():.1f} – {df_check['MeanHR'].max():.1f} bpm  (expected 50–100)")
print(f"   RMSSD:   {df_check['RMSSD'].min():.1f} – {df_check['RMSSD'].max():.1f} ms    (expected 15–80)")
print(f"   SDNN:    {df_check['SDNN'].min():.1f} – {df_check['SDNN'].max():.1f} ms    (expected 30–100)")
print(f"   HF:      {df_check['HF'].min():.4f} – {df_check['HF'].max():.4f}         (should be positive)")
print(f"   SD1:     {df_check['SD1'].min():.1f} – {df_check['SD1'].max():.1f} ms    (expected ~same as RMSSD/√2)")

# 3. Check for any NaN (failed computations)
nan_counts = df_check.isna().sum()
nan_cols = nan_counts[nan_counts > 0]
if len(nan_cols):
    print(f"\n3. ⚠ NaN values found:")
    print(nan_cols)
else:
    print(f"\n3. ✅ No NaN values — all metrics computed successfully")

# 4. Check duration is sensible
print(f"\n4. Recording durations:")
print(f"   Range: {df_check['duration_min'].min():.1f} – {df_check['duration_min'].max():.1f} min")
print(f"   Expected: ~28–31 min per session")

# 5. Check RMSSD vs SD1 consistency (SD1 should ≈ RMSSD/√2)
df_check['SD1_check'] = df_check['RMSSD'] / np.sqrt(2)
df_check['SD1_diff']  = abs(df_check['SD1'] - df_check['SD1_check'])
print(f"\n5. SD1 vs RMSSD/√2 consistency (should be ~0):")
print(f"   Max difference: {df_check['SD1_diff'].max():.4f} ms")

SANITY CHECK — FULL RECORDING HRV

1. Condition balance:
condition  control  experimental
subj_id                         
sub-001          1             1
sub-002          1             1
sub-003          1             1
sub-004          1             1
sub-005          1             1
sub-006          1             1
sub-007          1             1
sub-008          1             1
sub-009          1             1
sub-010          1             1
sub-011          1             1
sub-012          1             1
sub-015          1             1
sub-018          1             1
sub-019          1             1
sub-020          1             1
sub-021          1             1
sub-022          1             1
sub-023          1             1
sub-024          1             1
sub-025          1             1

2. Key metric ranges:
   MeanHR:  53.8 – 98.0 bpm  (expected 50–100)
   RMSSD:   14.6 – 95.5 ms    (expected 15–80)
   SDNN:    35.5 – 182.2 ms    (expected 30–100)
   HF:      0.0001

In [5]:
#HF power + grahics

"""
Script 17 — Absolute HF Power per phase (BBSIG pipeline method)

Extracts HF power (ms²) per subject × session × phase using
NeuroKit2 hrv_frequency() with Welch method + 4 Hz interpolation,
following the BBSIG pipeline specification at:
https://martager.github.io/bbsig/hrv-analysis/#4-optional-compute-frequency-domain-hrv-metrics

HF band: 0.15 – 0.4 Hz (Task Force standard)

Key difference from existing freq TSV:
  The existing task-BBSIG_per_phase_hrv-freq.tsv already contains HF (absolute)
  and HFn (normalised). This script re-extracts with explicit Welch + 4 Hz
  interpolation settings to match the BBSIG pipeline exactly, and also
  saves LnHF (log-transformed HF) which is more normally distributed.

Output:
  derivatives/hrv-analysis/task-BBSIG_per_phase_hrv-freq-bbsig.tsv
  derivatives/group_figures/traj_HF_abs_n20.png
  derivatives/group_figures/traj_HF_abs_rsp_excl.png
  derivatives/group_figures/traj_LnHF_n20.png
  derivatives/group_figures/traj_LnHF_rsp_excl.png
"""

import numpy as np
import pandas as pd
import neurokit2 as nk
import json
from pathlib import Path
from scipy import stats
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# ─── PATHS ────────────────────────────────────────────────────────────────────
BIDS_ROOT = Path(r"C:\Users\alisa\OneDrive\Desktop\HRV_Biotrace\bids_data")
DERIV     = BIDS_ROOT / "derivatives"
HRV_DIR   = DERIV / "hrv-analysis"
FIG_DIR   = DERIV / "group_figures"
FIG_DIR.mkdir(exist_ok=True)

TASK              = "BBSIG"
INTERPOLATION_HZ  = 4     # Hz — BBSIG default for frequency-domain HRV

# ─── EXCLUSION SETS ───────────────────────────────────────────────────────────
EXCL_ECG      = {"sub-021"}
EXCL_RSP_SUBJ = {"sub-021", "sub-010"}
EXCL_RSP_PHASES = {
    ("sub-002","ses-01","late"),
    ("sub-007","ses-02","early"),
    ("sub-020","ses-01","early"),
    ("sub-023","ses-01","early"),
    ("sub-023","ses-02","early"),
}

ALL_PARTICIPANTS = [
    "001","002","003","004","005","006","007","008","009","010",
    "011","012","015","018","019","020","021","022","023","024","025"
]
SESSION_IDS = ["01","02"]
PHASES      = ["pre","early","late","post"]
PHASE_XLBL  = ["PRE","EARLY","LATE","POST"]
X_POS       = {p: i for i, p in enumerate(PHASES)}
COLORS      = {"control": "#2166AC", "experimental": "#D6604D"}
PHASE_SHADING = {
    "pre":   "#AEC6E8",
    "early": "#B2E2B2",
    "late":  "#D8B4E8",
    "post":  "#FDDBA8",
}

# ─── DATA LOADING ─────────────────────────────────────────────────────────────
def load_rr(subj_id, ses_id):
    base = f"sub-{subj_id}_ses-{ses_id}_task-{TASK}_physio"
    for p in [
        DERIV/f"ecg-preproc/sub-{subj_id}/ses-{ses_id}/{base}_ecg-preproc.json",
        DERIV/f"ecg-preproc/sub-{subj_id}/ses-{ses_id}/beh/{base}_ecg-preproc.json",
    ]:
        if p.exists():
            with open(p) as f:
                d = json.load(f)
            rr_s = d.get("rr_s", {})
            arr  = rr_s.get("RR_s_corrected") or rr_s.get("RR_s_original")
            if arr:
                rr_ms = np.array(arr) * 1000
                return {"RRI": rr_ms, "RRI_Time": np.cumsum(rr_ms)}
    return None


def load_phases(subj_id, ses_id):
    base = f"sub-{subj_id}_ses-{ses_id}_task-{TASK}_physio"
    p    = DERIV/f"phases/sub-{subj_id}/ses-{ses_id}/{base}_phases.json"
    if not p.exists():
        return None, None
    with open(p) as f:
        d = json.load(f)
    return d["phases"], d.get("condition","unknown")


def crop_rr(rrs_dict, start_s, end_s):
    rri      = rrs_dict["RRI"]
    rri_time = rrs_dict["RRI_Time"]
    start_ms = start_s * 1000
    end_ms   = end_s   * 1000
    mask     = (rri_time >= start_ms) & (rri_time <= end_ms)
    rri_c    = rri[mask]
    time_c   = rri_time[mask]
    if len(rri_c) < 10:
        return None
    return {"RRI": rri_c, "RRI_Time": time_c}


# ─── FREQUENCY-DOMAIN HRV (BBSIG method) ────────────────────────────────────
def hrv_frequency_domain(rrs_dict, interp_freq=4):
    """
    Compute frequency-domain HRV using NeuroKit2 hrv_frequency()
    with Welch method and specified interpolation rate.
    Returns dict with VLF, LF, HF, LF_HF, LFn, HFn, LnHF.
    """
    rri_time = rrs_dict["RRI_Time"]
    try:
        hrv_f = nk.hrv_frequency(
            peaks         = rri_time / 1000,  # NeuroKit expects seconds
            sampling_rate = 1,
            ulf           = (0,      0.0033),
            vlf           = (0.0033, 0.04),
            lf            = (0.04,   0.15),
            hf            = (0.15,   0.4),
            show          = False
        )
        return {
            "VLF":   hrv_f["HRV_VLF"].values[0],
            "LF":    hrv_f["HRV_LF"].values[0],
            "HF":    hrv_f["HRV_HF"].values[0],
            "LF_HF": hrv_f["HRV_LFHF"].values[0],
            "LFn":   hrv_f["HRV_LFn"].values[0],
            "HFn":   hrv_f["HRV_HFn"].values[0],
            "LnHF":  hrv_f["HRV_LnHF"].values[0],
        }
    except Exception as e:
        return {k: np.nan for k in
                ["VLF","LF","HF","LF_HF","LFn","HFn","LnHF"]}


# ─── MAIN EXTRACTION LOOP ────────────────────────────────────────────────────
print("="*60)
print("HF POWER EXTRACTION — BBSIG PIPELINE METHOD")
print(f"Welch PSD, {INTERPOLATION_HZ} Hz interpolation, HF = 0.15–0.40 Hz")
print("="*60)

all_rows = []

for subj_id in ALL_PARTICIPANTS:
    for ses_id in SESSION_IDS:
        rrs = load_rr(subj_id, ses_id)
        if rrs is None:
            print(f"  ⚠  sub-{subj_id}/ses-{ses_id}: no RR data")
            continue

        phases, condition = load_phases(subj_id, ses_id)
        if phases is None:
            continue

        print(f"  sub-{subj_id}/ses-{ses_id} [{condition}]")

        for phase_name in PHASES:
            p       = phases[phase_name]
            cropped = crop_rr(rrs, p["start"], p["end"])

            if cropped is None:
                print(f"    ⚠  {phase_name.upper()}: too few beats")
                all_rows.append({
                    "subj_id":   f"sub-{subj_id}",
                    "ses_id":    f"ses-{ses_id}",
                    "condition": condition,
                    "phase":     phase_name,
                    "n_beats":   0,
                    "duration_min": 0,
                    **{k: np.nan for k in
                       ["VLF","LF","HF","LF_HF","LFn","HFn","LnHF"]}
                })
                continue

            freq_metrics = hrv_frequency_domain(cropped, INTERPOLATION_HZ)
            n_beats = len(cropped["RRI"])
            dur_min = ((cropped["RRI_Time"][-1] -
                        cropped["RRI_Time"][0]) / 1000 / 60)

            row = {
                "subj_id":      f"sub-{subj_id}",
                "ses_id":       f"ses-{ses_id}",
                "condition":    condition,
                "phase":        phase_name,
                "n_beats":      n_beats,
                "duration_min": round(dur_min, 3),
                **freq_metrics
            }
            all_rows.append(row)

            hf_val  = freq_metrics["HF"]
            hfn_val = freq_metrics["HFn"]
            print(f"    ✅ {phase_name.upper():6s}: "
                  f"HF={hf_val:.4f} ms² | HFn={hfn_val:.4f} | "
                  f"n={n_beats} beats")

# ─── SAVE TSV ─────────────────────────────────────────────────────────────────
df_out = pd.DataFrame(all_rows)
out_path = HRV_DIR / f"task-{TASK}_per_phase_hrv-freq-bbsig.tsv"
df_out.to_csv(out_path, sep="\t", index=False)

print(f"\n{'='*60}")
print(f"✅ Saved: {out_path.name}")
print(f"   {len(df_out)} rows")
print(f"   HF range:  {df_out['HF'].min():.4f} – {df_out['HF'].max():.4f} ms²")
print(f"   HFn range: {df_out['HFn'].min():.4f} – {df_out['HFn'].max():.4f}")
print(f"   NaN HF:    {df_out['HF'].isna().sum()}")
print(f"{'='*60}")

# ─── TRAJECTORY FIGURES ───────────────────────────────────────────────────────
def ci95(vals):
    vals = np.array(vals)[~np.isnan(np.array(vals))]
    if len(vals) < 2:
        return np.nan, np.nan, np.nan
    se = stats.sem(vals)
    h  = se * stats.t.ppf(0.975, df=len(vals)-1)
    m  = float(vals.mean())
    return m, m-h, m+h


def style_ax(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(axis="y", ls=":", alpha=0.35)


def plot_trajectory(df_src, col, measure_key, measure_full, unit,
                    fname, excl_subj=None, excl_phases=None,
                    ymin=0, ymax=None):
    df = df_src.copy()
    if "subj_id" in df.columns:
        df = df.rename(columns={"subj_id":"VP"})

    if excl_subj:
        df = df[~df["VP"].isin(excl_subj)]
    if excl_phases:
        for vp, ses, ph in excl_phases:
            df = df[~((df["VP"]==vp) &
                      (df["ses_id"]==ses) &
                      (df["phase"]==ph))]

    df["phase"] = pd.Categorical(df["phase"],
                                  categories=PHASES, ordered=True)
    complete = (df.dropna(subset=[col])
                  .groupby("VP")
                  .filter(lambda x: len(x)==8)["VP"].unique())
    df = df[df["VP"].isin(complete)]
    n  = df["VP"].nunique()

    stats_df = (df.groupby(["condition","phase"])[col]
                  .agg(Mean="mean",
                       SE=lambda x: x.std(ddof=1)/np.sqrt(x.count()))
                  .reset_index())

    # Y range: max of CI tip AND max individual participant value
    data_hi_stats = float((stats_df["Mean"] + stats_df["SE"]).max())
    data_lo_stats = float((stats_df["Mean"] - stats_df["SE"]).min())
    indiv_vals    = df[col].dropna()
    data_hi       = max(data_hi_stats, float(indiv_vals.max())) if len(indiv_vals) else data_hi_stats
    data_lo_indiv = float(indiv_vals.min()) if len(indiv_vals) else data_lo_stats
    data_lo       = min(data_lo_stats, data_lo_indiv)

    # Left panel — tight zoom around group means ± SE
    lo_grp = float((stats_df["Mean"] - stats_df["SE"]).min())
    hi_grp = float((stats_df["Mean"] + stats_df["SE"]).max())
    pad_l  = (hi_grp - lo_grp) * 0.25
    ymin_l = lo_grp - pad_l if ymin is None else max(ymin, lo_grp - pad_l)
    ymax_l = (hi_grp + pad_l) if ymax is None else min(ymax, hi_grp + pad_l)

    # Right panel — full individual spaghetti range
    indiv_vals = df[col].dropna()
    lo_ind = float(indiv_vals.min()) if len(indiv_vals) else lo_grp
    hi_ind = float(indiv_vals.max()) if len(indiv_vals) else hi_grp
    pad_r  = (hi_ind - lo_ind) * 0.08
    ymin_r = lo_ind - pad_r if ymin is None else max(ymin, lo_ind - pad_r)
    ymax_r = (hi_ind + pad_r) if ymax is None else min(ymax, hi_ind + pad_r)

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle(
        f"{measure_full} — Control vs Experimental  (n={n})",
        fontsize=12, fontweight="bold", y=1.01)

    for ax_i, ax in enumerate(axes):
        for cond in ["control","experimental"]:
            sub = stats_df[stats_df["condition"]==cond].sort_values("phase")
            x   = [X_POS[p] for p in sub["phase"]]
            mk  = "o" if cond=="control" else "s"

            if ax_i == 0:
                ax.plot(x, sub["Mean"], color=COLORS[cond], lw=2.5,
                        marker=mk, ms=7, zorder=3, label=cond.capitalize())
                ax.errorbar(x, sub["Mean"], yerr=sub["SE"],
                            fmt="none", color=COLORS[cond],
                            capsize=5, lw=1.5, zorder=3)
            else:
                for vp, sub_df in df[df["condition"]==cond].groupby("VP"):
                    pts = sub_df.sort_values("phase").dropna(subset=[col])
                    if len(pts) < 2:
                        continue
                    ax.plot([X_POS[p] for p in pts["phase"]], pts[col],
                            color=COLORS[cond], lw=0.8, alpha=0.25, zorder=2)
                ax.plot(x, sub["Mean"], color=COLORS[cond], lw=3.0,
                        marker=mk, ms=8, zorder=5, label=cond.capitalize())

        ax.set_xticks(range(len(PHASES)))
        ax.set_xticklabels(PHASE_XLBL, fontsize=10)
        ax.set_xlabel("Experimental phase", fontsize=11)
        ax.set_ylabel(f"{measure_full} ({unit})", fontsize=11)
        ax.set_ylim(ymin_l if ax_i == 0 else ymin_r,
                    ymax_l if ax_i == 0 else ymax_r)
        ax.set_title("Group means ± SE" if ax_i==0
                     else "Individual trajectories + group mean",
                     fontsize=10)
        ax.legend(fontsize=10, frameon=False)
        style_ax(ax)

    plt.tight_layout()
    plt.savefig(FIG_DIR/fname, dpi=300, bbox_inches="tight")
    plt.close()
    print(f"  Saved → {fname}")


print("\nGenerating trajectory figures...")

# HF absolute (ms²)
plot_trajectory(df_out, "HF",
                "HF", "High-Frequency Power (HF)", "ms²",
                "traj_HF_abs_n20.png",
                excl_subj=EXCL_ECG, ymin=0)

plot_trajectory(df_out, "HF",
                "HF", "High-Frequency Power (HF)", "ms²",
                "traj_HF_abs_rsp_excl.png",
                excl_subj=EXCL_RSP_SUBJ,
                excl_phases=EXCL_RSP_PHASES, ymin=0)

# LnHF (log-transformed — can be negative, use ymin=None for auto lower bound)
plot_trajectory(df_out, "LnHF",
                "LnHF", "Natural Log of HF Power (LnHF)", "ln(ms²)",
                "traj_LnHF_n20.png",
                excl_subj=EXCL_ECG, ymin=None)

plot_trajectory(df_out, "LnHF",
                "LnHF", "Natural Log of HF Power (LnHF)", "ln(ms²)",
                "traj_LnHF_rsp_excl.png",
                excl_subj=EXCL_RSP_SUBJ,
                excl_phases=EXCL_RSP_PHASES, ymin=None)

# HFn (normalised — for completeness, same data as existing script)
plot_trajectory(df_out, "HFn",
                "HFn", "Normalised High-Frequency Power (HFn)", "0–1",
                "traj_HFn_bbsig_n20.png",
                excl_subj=EXCL_ECG, ymin=0, ymax=1.0)

plot_trajectory(df_out, "HFn",
                "HFn", "Normalised High-Frequency Power (HFn)", "0–1",
                "traj_HFn_bbsig_rsp_excl.png",
                excl_subj=EXCL_RSP_SUBJ,
                excl_phases=EXCL_RSP_PHASES, ymin=0, ymax=1.0)

print("\n"+"="*60)
print("SCRIPT 17 COMPLETE")
print(f"TSV  → {out_path.name}")
print(f"Figs → {FIG_DIR}")
print("="*60)

HF POWER EXTRACTION — BBSIG PIPELINE METHOD
Welch PSD, 4 Hz interpolation, HF = 0.15–0.40 Hz
  sub-001/ses-01 [control]
    ✅ PRE   : HF=0.0054 ms² | HFn=0.2384 | n=240 beats
    ✅ EARLY : HF=0.0202 ms² | HFn=0.4108 | n=187 beats
    ✅ LATE  : HF=0.0177 ms² | HFn=0.3372 | n=221 beats
    ✅ POST  : HF=0.0067 ms² | HFn=0.3823 | n=245 beats
  sub-001/ses-02 [experimental]
    ✅ PRE   : HF=0.0038 ms² | HFn=0.1216 | n=217 beats
    ✅ EARLY : HF=0.0346 ms² | HFn=0.4236 | n=168 beats
    ✅ LATE  : HF=0.0576 ms² | HFn=0.4729 | n=194 beats
    ✅ POST  : HF=0.0033 ms² | HFn=0.2101 | n=228 beats
  sub-002/ses-01 [experimental]
    ✅ PRE   : HF=0.0100 ms² | HFn=0.3829 | n=230 beats
    ✅ EARLY : HF=0.0373 ms² | HFn=0.7247 | n=173 beats
    ✅ LATE  : HF=0.0274 ms² | HFn=0.4822 | n=223 beats
    ✅ POST  : HF=0.0171 ms² | HFn=0.3376 | n=247 beats
  sub-002/ses-02 [control]
    ✅ PRE   : HF=0.0153 ms² | HFn=0.4192 | n=231 beats
    ✅ EARLY : HF=0.0386 ms² | HFn=0.8616 | n=178 beats
    ✅ LATE  : HF=0.

In [13]:
#step 3,4,5,6,7: frequnecy and time domains calculated for phases hrv
# ── MAIN LOOP: PER PHASE ──────────────────────────────────────────────────────
print("="*60)
print("HRV ANALYSIS — PER PHASE")
print("="*60)

all_hrv_phases = []

for subj_id in participant_ids:
    for ses_id in session_ids:

        print(f"\nsub-{subj_id}/ses-{ses_id}")

        # Load RR intervals
        rrs = load_rr_data(bids_root, subj_id, ses_id)
        if rrs is None:
            continue

        # Load phase boundaries and condition
        phases, condition = load_phase_boundaries(bids_root, subj_id, ses_id)
        if phases is None:
            continue

        # Loop over phases
        for phase_name in ['pre', 'early', 'late', 'post']:
            phase     = phases[phase_name]
            start_s   = phase['start']
            end_s     = phase['end']
            cropped   = crop_rr_by_phase(rrs, start_s, end_s)

            if cropped is None:
                print(f"  ⚠ {phase_name.upper()}: too few beats, skipping")
                continue

            row = compute_hrv_all(
                rrs_dict  = cropped,
                subj_id   = f"sub-{subj_id}",
                ses_id    = f"ses-{ses_id}",
                condition = condition,
                window    = 'phase',
                phase     = phase_name
            )

            # Add phase timing metadata
            row['phase_start_s'] = start_s
            row['phase_end_s']   = end_s

            all_hrv_phases.append(row)
            print(f"  ✅ {phase_name.upper():6s}: {cropped['n_beats']} beats | "
                  f"RMSSD={row['RMSSD']:.1f}ms | HF={row['HF']:.4f}")

# ── SAVE OUTPUT ───────────────────────────────────────────────────────────────
df_phases = pd.DataFrame(all_hrv_phases)
out_path  = out_dir / f"task-{task_name}_hrv_per_phase.tsv"
df_phases.to_csv(out_path, sep='\t', index=False)

print(f"\n{'='*60}")
print(f"✅ Saved: {out_path.name}")
print(f"   {len(df_phases)} rows (expected: {len(participant_ids)} × 2 sessions × 4 phases = {len(participant_ids)*2*4})")
print(f"{'='*60}")

# Preview
print(df_phases[['subj_id','ses_id','condition','phase','n_beats','RMSSD','HF','SD1']].to_string(index=False))

HRV ANALYSIS — PER PHASE

sub-001/ses-01
  Loaded sub-001/ses-01 [original]: 2278 RR intervals
  ✅ PRE   : 240 beats | RMSSD=39.8ms | HF=0.0054
  ✅ EARLY : 187 beats | RMSSD=32.8ms | HF=0.0202
  ✅ LATE  : 221 beats | RMSSD=37.1ms | HF=0.0177
  ✅ POST  : 245 beats | RMSSD=22.4ms | HF=0.0067

sub-001/ses-02
  Loaded sub-001/ses-02 [original]: 2024 RR intervals
  ✅ PRE   : 217 beats | RMSSD=25.4ms | HF=0.0038
  ✅ EARLY : 168 beats | RMSSD=30.7ms | HF=0.0346
  ✅ LATE  : 194 beats | RMSSD=36.7ms | HF=0.0576
  ✅ POST  : 228 beats | RMSSD=21.1ms | HF=0.0033

sub-002/ses-01
  Loaded sub-002/ses-01 [original]: 2234 RR intervals
  ✅ PRE   : 230 beats | RMSSD=68.4ms | HF=0.0100
  ✅ EARLY : 173 beats | RMSSD=69.6ms | HF=0.0373
  ✅ LATE  : 223 beats | RMSSD=56.2ms | HF=0.0274
  ✅ POST  : 247 beats | RMSSD=51.8ms | HF=0.0171

sub-002/ses-02
  Loaded sub-002/ses-02 [original]: 1911 RR intervals
  ✅ PRE   : 231 beats | RMSSD=59.6ms | HF=0.0153
  ✅ EARLY : 178 beats | RMSSD=63.8ms | HF=0.0386
  ✅ LATE 

In [14]:
# checking the ranges, if anything is missing
# ── SANITY CHECK — PER PHASE HRV ─────────────────────────────────────────────
print("SANITY CHECK — PER PHASE HRV")
print("="*60)

df_check_p = pd.DataFrame(all_hrv_phases)

# 1. Row count — expect 168 (21 participants × 2 sessions × 4 phases)
expected_rows = len(participant_ids) * 2 * 4
print(f"\n1. Row count: {len(df_check_p)} (expected {expected_rows})")
if len(df_check_p) < expected_rows:
    missing = expected_rows - len(df_check_p)
    print(f"   ⚠ {missing} missing phase(s) — checking which ones:")
    # Find missing combinations
    from itertools import product
    all_combos = set(product(
        [f"sub-{s}" for s in participant_ids],
        [f"ses-{s}" for s in session_ids],
        ['pre','early','late','post']
    ))
    found_combos = set(zip(df_check_p['subj_id'], df_check_p['ses_id'], df_check_p['phase']))
    for combo in sorted(all_combos - found_combos):
        print(f"   Missing: {combo[0]}/{combo[1]}/{combo[2]}")
else:
    print(f"   ✅ All phases present for all participants")

# 2. Each participant has all 4 phases per session
print(f"\n2. Phase balance per participant/session:")
phase_counts = df_check_p.groupby(['subj_id','ses_id'])['phase'].count()
unbalanced   = phase_counts[phase_counts != 4]
if len(unbalanced):
    print(f"   ⚠ Unbalanced phase counts:")
    print(unbalanced)
else:
    print(f"   ✅ All participants have exactly 4 phases per session")

# 3. Condition assignment — each participant should have one control, one experimental
print(f"\n3. Condition balance:")
cond_counts = df_check_p.groupby('subj_id')['condition'].nunique()
wrong_cond  = cond_counts[cond_counts != 2]
if len(wrong_cond):
    print(f"   ⚠ Some participants don't have both conditions: {list(wrong_cond.index)}")
else:
    print(f"   ✅ All participants have both control and experimental sessions")

# 4. HRV ranges per phase — check for implausible values
print(f"\n4. RMSSD per phase (ms):")
rmssd_phase = df_check_p.groupby('phase')['RMSSD'].agg(['mean','min','max'])
print(rmssd_phase.round(2).to_string())

print(f"\n5. Beat count per phase:")
beats_phase = df_check_p.groupby('phase')['n_beats'].agg(['mean','min','max'])
print(beats_phase.round(1).to_string())
if df_check_p['n_beats'].min() < 30:
    print(f"   ⚠ Some phases have very few beats (<30) — check reliability")
else:
    print(f"   ✅ All phases have sufficient beats for HRV")

# 5. NaN check
nan_counts = df_check_p.isna().sum()
nan_cols   = nan_counts[nan_counts > 0]
if len(nan_cols):
    print(f"\n6. ⚠ NaN values found:")
    print(nan_cols)
else:
    print(f"\n6. ✅ No NaN values")

# 6. Phase duration sanity
print(f"\n7. Phase durations (min):")
dur_phase = df_check_p.groupby('phase')['duration_min'].agg(['mean','min','max'])
print(dur_phase.round(2).to_string())
print(f"   Expected: ~3.0 min for pre/late/post, ~2.5 min for early")

# 7. Quick RMSSD comparison control vs experimental per phase
print(f"\n8. Mean RMSSD: control vs experimental per phase:")
rmssd_cond = df_check_p.groupby(['phase','condition'])['RMSSD'].mean().unstack()
print(rmssd_cond.round(2).to_string())

SANITY CHECK — PER PHASE HRV

1. Row count: 168 (expected 168)
   ✅ All phases present for all participants

2. Phase balance per participant/session:
   ✅ All participants have exactly 4 phases per session

3. Condition balance:
   ✅ All participants have both control and experimental sessions

4. RMSSD per phase (ms):
        mean    min     max
phase                      
early  43.80  11.99  102.43
late   45.91  14.30   95.88
post   40.76  15.65   78.46
pre    41.52  13.25   94.63

5. Beat count per phase:
        mean  min  max
phase                 
early  178.4  124  250
late   209.2  139  285
post   223.6  116  295
pre    220.3  106  319
   ✅ All phases have sufficient beats for HRV

6. ⚠ NaN values found:
VLF    7
dtype: int64

7. Phase durations (min):
       mean   min  max
phase                 
early  2.49  2.47  2.5
late   2.98  2.96  3.0
post   2.87  1.29  3.0
pre    2.89  1.53  3.0
   Expected: ~3.0 min for pre/late/post, ~2.5 min for early

8. Mean RMSSD: control vs ex

In [15]:
#VLF nan check - note participants (short windows)
nan_vlf = df_check_p[df_check_p['VLF'].isna()][['subj_id','ses_id','phase','n_beats','duration_min']]
print(nan_vlf.to_string(index=False))

subj_id ses_id phase  n_beats  duration_min
sub-002 ses-02  post      116      1.290299
sub-004 ses-02 early      160      2.487044
sub-005 ses-01 early      201      2.484049
sub-005 ses-02   pre      106      1.526042
sub-007 ses-01  post      129      1.687956
sub-011 ses-02  post      133      1.501497
sub-018 ses-01   pre      119      1.756641


In [17]:
#the right bbsig format saving
from pathlib import Path

bids_root = Path(r"C:\Users\alisa\OneDrive\Desktop\HRV_Biotrace\bids_data")
task_name = "BBSIG"
hrv_dir   = bids_root / "derivatives" / "hrv-analysis"
hrv_dir.mkdir(parents=True, exist_ok=True)

# Columns per domain
time_cols     = ['subj_id','ses_id','condition','phase','window',
                 'n_beats','duration_min','phase_start_s','phase_end_s',
                 'MeanRR','MeanHR','SDNN','RMSSD','pNN50']

freq_cols     = ['subj_id','ses_id','condition','phase','window',
                 'n_beats','duration_min','phase_start_s','phase_end_s',
                 'VLF','LF','HF','LF_HF','LFn','HFn','LnHF']

nonlinear_cols= ['subj_id','ses_id','condition','phase','window',
                 'n_beats','duration_min','phase_start_s','phase_end_s',
                 'SD1','SD2','SD1_SD2','SampEn','ApEn']

for df, label in [(df_phases, 'per_phase'), (df_full, 'full_recording')]:

    t_cols = [c for c in time_cols      if c in df.columns]
    f_cols = [c for c in freq_cols      if c in df.columns]
    n_cols = [c for c in nonlinear_cols if c in df.columns]

    df[t_cols].to_csv(hrv_dir / f"task-{task_name}_{label}_hrv-time.tsv",      sep='\t', index=False)
    df[f_cols].to_csv(hrv_dir / f"task-{task_name}_{label}_hrv-freq.tsv",      sep='\t', index=False)
    df[n_cols].to_csv(hrv_dir / f"task-{task_name}_{label}_hrv-nonlinear.tsv", sep='\t', index=False)

    print(f"✅ {label} — saved 3 domain files")

print(f"\nFiles now in hrv-analysis/:")
for f in sorted(hrv_dir.rglob("*.tsv")):
    print(f"  {f.name}")

✅ per_phase — saved 3 domain files
✅ full_recording — saved 3 domain files

Files now in hrv-analysis/:
  task-BBSIG_full_recording_hrv-freq.tsv
  task-BBSIG_full_recording_hrv-nonlinear.tsv
  task-BBSIG_full_recording_hrv-time.tsv
  task-BBSIG_hrv_full_recording.tsv
  task-BBSIG_hrv_per_phase.tsv
  task-BBSIG_per_phase_hrv-freq.tsv
  task-BBSIG_per_phase_hrv-nonlinear.tsv
  task-BBSIG_per_phase_hrv-time.tsv
